### 0-Setup

In [1]:
import os
from pathlib import Path

# Move from "src/notebook/ to the root to simplify import
os.chdir(Path.cwd().parents[0])
os.chdir(Path.cwd().parents[0])
print(f"📁 Current directory : {Path.cwd()}")

from src.config.constants import PROJECT_ROOT

print(PROJECT_ROOT)

📁 Current directory : c:\Users\MSI\Desktop\SB\Code\data_engineering_portfolio
C:\Users\MSI\Desktop\SB\Code\data_engineering_portfolio


### 1-Raw Data Generation with Faker

In [2]:
from src.config.constants import RUN_DATE, RAW_DATA_PATH, RAW_FILES
from pathlib import Path

import pandas as pd
import random
from faker import Faker
from datetime import timedelta

def build_local_path(table_name: str) -> Path:
    path = Path(RAW_DATA_PATH) / table_name / f"run_date={RUN_DATE}"
    path.mkdir(parents=True, exist_ok=True)

    return path

def write_parquet(df: pd.DataFrame, table_name: str) -> None:
    local_path = build_local_path(table_name)
    file_path = local_path / RAW_FILES[table_name]

    df.to_parquet(str(file_path), index=False)

In [3]:
# ---------- 0. Setup ----------
fake = Faker()

# ---------- 1. Customers ----------
num_customers = 50
customers = [
    {
        "customer_id": f"C{i:03d}",
        "name": fake.name(),
        "email": fake.email(),
        "country": fake.country(),
        "signup_date": fake.date_between(start_date="-2y", end_date="today")
    }
    for i in range(1, num_customers + 1)
]
df_customers = pd.DataFrame(customers)
write_parquet(df_customers, "customers")

# ---------- 2. Products ----------
categories = ["Electronics", "Clothing", "Books", "Sports"]
num_products = 20
products = []

for i in range(1, num_products + 1):
    category = categories[i % len(categories)]
    name = f"{category} Product {i:02d}" 
    cost = round(random.uniform(5, 50), 2)
    price = round(random.uniform(20, 200), 2)
    products.append({
        "product_id": f"P{i:03d}",
        "category": category,
        "name": name,
        "cost": cost,
        "price": price
    })

df_products = pd.DataFrame(products)
write_parquet(df_products, "products")

# ---------- 3. Orders ----------
num_orders = 200
orders = []

for i in range(1, num_orders + 1):
    product = random.choice(products)
    customer = random.choice(customers)
    quantity = random.randint(1, 5)
    orders.append({
        "order_id": f"O{RUN_DATE.replace('-', '')}{i:03d}",
        "order_date": fake.date_between(start_date="-1y", end_date="today"),
        "run_date": RUN_DATE,
        "customer_id": customer["customer_id"],
        "product_id": product["product_id"],
        "quantity": quantity,
        "price": product["price"],
        "country": customer["country"]
    })
df_orders = pd.DataFrame(orders)
write_parquet(df_orders, "orders")

# ---------- 4. Payments ----------
payments = [
    {
        "payment_id": f"PAY{i:03d}",
        "order_id": o["order_id"],
        "payment_method": random.choice(["Credit Card", "PayPal", "Bank Transfer"]),
        "payment_date": o["order_date"] + timedelta(days=random.randint(0, 2)),
        "amount": round(o["quantity"] * o["price"], 2)
    }
    for i, o in enumerate(orders, start=1)
]
df_payments = pd.DataFrame(payments)
write_parquet(df_payments, "payments")

print("✅ Local Parquet files generated successfully.")


✅ Local Parquet files generated successfully.


### 2-Raw Data Ingestion into GCS

In [4]:
import os
from pathlib import Path
from google.cloud import storage
from google.cloud.storage import Bucket
from src.config.constants import RUN_DATE, KEY_PATH, GCS_BUCKET_NAME, RAW_DATA_PATH, RAW_FILES

# ---------- 0. Setup ----------
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = KEY_PATH

# ---------- 1. Build GCS client ----------
client = storage.Client()
bucket = client.get_bucket(GCS_BUCKET_NAME)

# ---------- 2. Upload to GCS ----------
def upload_to_gcs(local_file: str, table_name: str, bucket: Bucket) -> None:
    """
    Upload a local Parquet file to GCS using the path structure:
    gs://<bucket>/raw/<table_name>/run_date=<run_date>/<file>
    """
    # Blob path in GCS
    gcs_blob = f"raw/{table_name}/run_date={RUN_DATE}/{os.path.basename(local_file)}"

    blob = bucket.blob(gcs_blob)
    blob.upload_from_filename(local_file)
    print(f"✅ Uploaded : gs://{GCS_BUCKET_NAME}/{gcs_blob}")

# ---------- 3. Upload all tables ----------
for table_name, filename in RAW_FILES.items():
    local_file = Path(RAW_DATA_PATH) / table_name / f"run_date={RUN_DATE}" / filename
    upload_to_gcs(str(local_file), table_name, bucket)


✅ Uploaded : gs://data-engineering-portfolio-bucket/raw/customers/run_date=2025-11-20/customers.parquet
✅ Uploaded : gs://data-engineering-portfolio-bucket/raw/products/run_date=2025-11-20/products.parquet
✅ Uploaded : gs://data-engineering-portfolio-bucket/raw/orders/run_date=2025-11-20/orders.parquet
✅ Uploaded : gs://data-engineering-portfolio-bucket/raw/payments/run_date=2025-11-20/payments.parquet


### 3-Raw Data Processing with Spark

In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.types import DecimalType
from pyspark.sql.functions import col, sum as _sum, count, lit
from src.config.constants import (
    RUN_DATE,
    KEY_PATH, 
    JAR_GCS_CONNECTOR, 
    RAW_PREFIX, 
    RAW_FILES,
    PROCESSED_PREFIX
)
import logging

# ---------- 0. Setup ----------
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = KEY_PATH

# ---------- 1. Spark session ----------
spark = SparkSession.builder \
    .appName("DataEngineeringPortfolio") \
    .config("spark.jars", JAR_GCS_CONNECTOR) \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")  #Remove log messages

# ---------- 2. Hadoop config ----------
hconf = spark._jsc.hadoopConfiguration()
hconf.set("fs.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem")
hconf.set("fs.AbstractFileSystem.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFS")
hconf.set("google.cloud.auth.service.account.enable", "true")
hconf.set("google.cloud.auth.service.account.json.keyfile", KEY_PATH)

# ---------- 3. Paths ----------
paths = {
    f: f"{RAW_PREFIX}{f}/run_date={RUN_DATE}/*.parquet"
    for f in RAW_FILES
}
# ---------- 4. Load data ----------
customers_df = spark.read.option("header", True).parquet(paths["customers"]).alias("c")
products_df = spark.read.option("header", True).parquet(paths["products"]).alias("p")
orders_df = spark.read.option("header", True).parquet(paths["orders"]).alias("o")
payments_df = spark.read.option("header", True).parquet(paths["payments"])

print("✅ Raw Parquet files loaded from GCS")

# ---------- 4. Rename columns to avoid duplicates and enforce uniqueness ----------
orders_clean = orders_df \
    .dropDuplicates(["order_id"]) \
    .withColumnRenamed("quantity", "order_quantity") \
    .withColumnRenamed("price", "order_price") \
    .withColumnRenamed("country", "order_country")

customers_clean = customers_df \
    .dropDuplicates(["customer_id"]) \
    .withColumnRenamed("name", "customer_name") \
    .withColumnRenamed("country", "customer_country")

products_clean = products_df \
    .dropDuplicates(["product_id"]) \
    .withColumnRenamed("name", "product_name")

# ---------- 5. Enrich orders with customer and product info ----------
orders_enriched = orders_clean.alias("o") \
    .join(customers_clean.alias("c"), on="customer_id", how="left") \
    .join(products_clean.alias("p"), on="product_id", how="left") \
    .withColumn("order_quantity", col("order_quantity").cast("double")) \
    .withColumn("order_price", col("order_price").cast("double")) \
    .withColumn("total_amount", (col("order_quantity") * col("order_price")).cast(DecimalType(18,2))) \
    .withColumn("run_date", lit(RUN_DATE))

# ---------- 6. Aggregate revenue per customer ----------
customers_revenue = (
    orders_enriched
    .groupBy("customer_id", "customer_name", "run_date")
    .agg(_sum(col("total_amount").cast(DecimalType(18,2))).alias("total_revenue"))
)

# ---------- 7. Aggregate sales per product ----------
products_sales = (
    orders_enriched
    .groupBy("product_id", "product_name", "category", "run_date")
    .agg(
        _sum("order_quantity").alias("total_quantity_sold"),
        _sum("total_amount").alias("total_revenue")
    )
)

# ---------- 8. Aggregate revenue per category ----------
category_revenue = orders_enriched \
    .groupBy("category", "run_date") \
    .agg(_sum("total_amount").alias("category_revenue"))
    
# ---------- 9. Aggregate payment summary ----------
payments_summary = payments_df.withColumn("amount", col("amount").cast("double")) \
    .withColumn("run_date", lit(RUN_DATE)) \
    .groupBy("run_date", "payment_method") \
    .agg(
        _sum("amount").alias("total_amount"),
        count("amount").alias("total_count")
    )

print("✅ Spark aggregations complete")

# ---------- 10. Save processed tables in GCS ----------
outputs = {
    "orders_enriched": orders_enriched,
    "customers_revenue": customers_revenue,
    "products_sales": products_sales,
    "category_revenue": category_revenue,
    "payments_summary": payments_summary
}
    
for name, df in outputs.items():
    output_path = f"{PROCESSED_PREFIX}{name}/run_date={RUN_DATE}/"
    df.write.option("header", True).mode("overwrite").parquet(output_path)
    print(f"✅ Saved {name} → {output_path}")
    
print("✅ All processed Parquet files saved successfully.")


✅ Raw Parquet files loaded from GCS
✅ Spark aggregations complete
✅ Saved orders_enriched → gs://data-engineering-portfolio-bucket/processed/orders_enriched/run_date=2025-11-20/
✅ Saved customers_revenue → gs://data-engineering-portfolio-bucket/processed/customers_revenue/run_date=2025-11-20/
✅ Saved products_sales → gs://data-engineering-portfolio-bucket/processed/products_sales/run_date=2025-11-20/
✅ Saved category_revenue → gs://data-engineering-portfolio-bucket/processed/category_revenue/run_date=2025-11-20/
✅ Saved payments_summary → gs://data-engineering-portfolio-bucket/processed/payments_summary/run_date=2025-11-20/
✅ All processed Parquet files saved successfully.


In [6]:
orders_enriched.show(5)

+----------+-----------+------------+----------+----------+--------------+-----------+-------------+-----------------+--------------------+----------------+-----------+--------+-------------------+-----+-----+------------------+
|product_id|customer_id|    order_id|order_date|  run_date|order_quantity|order_price|order_country|    customer_name|               email|customer_country|signup_date|category|       product_name| cost|price|      total_amount|
+----------+-----------+------------+----------+----------+--------------+-----------+-------------+-----------------+--------------------+----------------+-----------+--------+-------------------+-----+-----+------------------+
|      P006|       C006|O20251120001|2025-04-11|2025-11-20|           2.0|      37.35|       Mexico| Melanie Stephens|dawsonmolly@examp...|          Mexico| 2025-03-08|   Books|   Books Product 06|46.56|37.35|              74.7|
|      P013|       C026|O20251120002|2025-01-25|2025-11-20|           4.0|       39.

In [7]:
category_revenue.show(5)

+-----------+----------+------------------+
|   category|  run_date|  category_revenue|
+-----------+----------+------------------+
|      Books|2025-11-20|13851.960000000003|
|   Clothing|2025-11-20|15835.079999999994|
|     Sports|2025-11-20|11030.059999999996|
|Electronics|2025-11-20|23583.619999999984|
+-----------+----------+------------------+



### 4-GCS Processed Data loading into BigQuery

In [8]:
import os
from google.cloud import bigquery
from src.config.constants import (
    RUN_DATE,
    KEY_PATH,
    PROJECT_ID,
    BQ_DATASET_NAME,
    REGION,
    PROCESSED_PREFIX,
    PROCESSED_TABLES,
)

# ---------- 0. Setup ----------
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = KEY_PATH

# ---------- 1. BigQuery client ----------
bq_client = bigquery.Client(project=PROJECT_ID)

# ---------- 2. Dataset ----------
dataset_id = f"{PROJECT_ID}.{BQ_DATASET_NAME}"
dataset = bigquery.Dataset(dataset_id)
dataset.location = REGION
bq_client.create_dataset(dataset, exists_ok=True)

# ---------- 3. Write Disposition Rules ----------
WRITE_MODES = {
    "orders_enriched": "WRITE_APPEND",   # <- Fact table
    "customers_revenue": "WRITE_APPEND",
    "products_sales": "WRITE_APPEND",
    "category_revenue": "WRITE_APPEND",
    "payments_summary": "WRITE_APPEND",
}

# ---------- 4. Load processed tables ----------
for table_name in PROCESSED_TABLES:

    gcs_uri = f"{PROCESSED_PREFIX}{table_name}/run_date={RUN_DATE}/*.parquet"
    table_id = f"{dataset_id}.{table_name}"

    write_mode = WRITE_MODES.get(table_name, "WRITE_TRUNCATE")

    job_config = bigquery.LoadJobConfig(
        source_format=bigquery.SourceFormat.PARQUET,
        autodetect=True,
        write_disposition=write_mode
    )

    load_job = bq_client.load_table_from_uri(
        gcs_uri,
        table_id,
        job_config=job_config
    )
    load_job.result()

    print(f"✅ Loaded {table_name} into BigQuery")

# ---------- 5. Schema Cleanup for category_revenue ----------
final_table_name = "category_revenue_clean"
source_table_name = "category_revenue"

clean_copy_query = f"""
CREATE OR REPLACE TABLE `{dataset_id}.{final_table_name}` AS
SELECT
category,
SAFE_CAST(category_revenue AS FLOAT64) AS category_revenue
FROM `{dataset_id}.{source_table_name}`;
"""

query_job = bq_client.query(clean_copy_query)
query_job.result()

print(f"✅ Clean table created: {final_table_name}")


✅ Loaded orders_enriched into BigQuery
✅ Loaded customers_revenue into BigQuery
✅ Loaded products_sales into BigQuery
✅ Loaded category_revenue into BigQuery
✅ Loaded payments_summary into BigQuery
✅ Clean table created: category_revenue_clean


### 5-BigQuery Validation

In [9]:
import os
from google.cloud import bigquery
from decimal import Decimal, getcontext
from typing import List, Dict, Any, Union
from src.config.constants import KEY_PATH, PROJECT_ID, BQ_DATASET_NAME, VERBOSE

Number = Union[int, float, str, Decimal]

# ---------- 0. Setup ----------
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = KEY_PATH
client = bigquery.Client(project=PROJECT_ID)
dataset_id = f"{PROJECT_ID}.{BQ_DATASET_NAME}"

EPSILON = Decimal("0.01")
getcontext().prec = 6  # precision

def run_query(query: str) -> List[Dict[str, Any]]:
    query_job = client.query(query)
    return [dict(row) for row in query_job.result()]

def check_diff(a: Number, b: Number) -> bool:
    return abs(Decimal(a) - Decimal(b)) <= EPSILON

def run_test(name: str, query: str, check_fn=None, verbose: bool=VERBOSE) -> List[Dict[str, Any]]:
    result = run_query(query)

    if check_fn and not check_fn(result):
            print(f"❌ {name} failed: {result}")
            raise AssertionError(f"{name} failed: {result}")
    if verbose:
        print(f"✅ {name} : {result or None}")

    return result

# ---------- 1. Orders & quantities ----------
run_test(
    "Test 1: Orders & quantities",
    f"""
    SELECT
        COUNT(DISTINCT order_id) AS n_orders,
        SUM(order_quantity) AS total_quantity
    FROM `{dataset_id}.orders_enriched`
    """,
    lambda r: r[0]["n_orders"] <= r[0]["total_quantity"]
)

# ---------- 2. Customers & products existence ----------
run_test(
    "Test 2: Customers & products existence",
    f"""
    SELECT
        COUNT(DISTINCT customer_id) AS n_customers,
        COUNT(DISTINCT product_id) AS n_products
    FROM `{dataset_id}.orders_enriched`
    """,
    lambda r: r[0]["n_customers"] > 0 and r[0]["n_products"] > 0
)

# ---------- 3. Product name consistency ----------
run_test(
    "Test 3: Product name consistency",
    f"""
    SELECT product_id, COUNT(DISTINCT product_name) AS cnt
    FROM `{dataset_id}.orders_enriched`
    GROUP BY product_id
    HAVING cnt > 1
    """,
    lambda r: len(r) == 0
)

# ---------- 4. Missing joins ----------
run_test(
    "Test 4: Missing customers/products",
    f"""
    SELECT COUNT(*) AS missing
    FROM `{dataset_id}.orders_enriched`
    WHERE customer_id IS NULL OR product_id IS NULL
    """,
    lambda r: r[0]["missing"] == 0
)

# ---------- 5. Duplicate orders ----------
run_test(
    "Test 5: Duplicate orders",
    f"""
    SELECT order_id, COUNT(*) AS cnt
    FROM `{dataset_id}.orders_enriched`
    GROUP BY order_id
    HAVING cnt > 1
    """,
    lambda r: len(r) == 0
)

# ---------- 6. Global revenue consistency ----------
run_test(
    "Test 6: Global revenue consistency",
    f"""
    SELECT
        (SELECT SUM(total_amount) FROM `{dataset_id}.orders_enriched`) AS orders_total,
        (SELECT SUM(total_revenue) FROM `{dataset_id}.customers_revenue`) AS customers_total,
        (SELECT SUM(total_revenue) FROM `{dataset_id}.products_sales`) AS products_total
    """,
    lambda r: check_diff(r[0]["orders_total"], r[0]["customers_total"]) and check_diff(r[0]["orders_total"], r[0]["products_total"])
)

# ---------- 7. Customer aggregation correctness ----------
run_test(
    "Test 7: Customer aggregation correctness",
    f"""
    WITH recalculated AS (
        SELECT customer_id, run_date, SUM(total_amount) AS true_revenue
        FROM `{dataset_id}.orders_enriched`
        GROUP BY customer_id, run_date
    )
    SELECT
        SUM(r.true_revenue) AS recomputed,
        SUM(c.total_revenue) AS stored
    FROM recalculated r
    JOIN `{dataset_id}.customers_revenue` c
    ON r.customer_id = c.customer_id
    AND r.run_date = c.run_date
    """,
    lambda r: check_diff(r[0]["recomputed"], r[0]["stored"])
)

# ---------- 8. Product aggregation correctness ----------
run_test(
    "Test 8: Product aggregation correctness",
    f"""
    WITH recalculated AS (
        SELECT product_id, run_date, SUM(order_quantity) AS true_quantity, SUM(total_amount) AS true_revenue
        FROM `{dataset_id}.orders_enriched`
        GROUP BY product_id, run_date
    )
    SELECT
        SUM(r.true_quantity) AS recomputed_qty,
        SUM(p.total_quantity_sold) AS stored_qty,
        SUM(r.true_revenue) AS recomputed_rev,
        SUM(p.total_revenue) AS stored_rev
    FROM recalculated r
    JOIN `{dataset_id}.products_sales` p
    ON r.product_id = p.product_id
    AND r.run_date = p.run_date
    """,
    lambda r: check_diff(r[0]["recomputed_qty"], r[0]["stored_qty"]) and check_diff(r[0]["recomputed_rev"], r[0]["stored_rev"])
)

print("✅ All BigQuery Data Validation passed")


✅ Test 1: Orders & quantities : [{'n_orders': 200, 'total_quantity': 571.0}]
✅ Test 2: Customers & products existence : [{'n_customers': 49, 'n_products': 20}]
✅ Test 3: Product name consistency : None
✅ Test 4: Missing customers/products : [{'missing': 0}]
✅ Test 5: Duplicate orders : None
✅ Test 6: Global revenue consistency : [{'orders_total': 64300.71999999998, 'customers_total': 64300.71999999997, 'products_total': 64300.719999999994}]
✅ Test 7: Customer aggregation correctness : [{'recomputed': 64300.71999999998, 'stored': 64300.71999999998}]
✅ Test 8: Product aggregation correctness : [{'recomputed_qty': 571.0, 'stored_qty': 571.0, 'recomputed_rev': 64300.719999999994, 'stored_rev': 64300.719999999994}]
✅ All BigQuery Data Validation passed
